# Fixing an LLM-generated spec with `check_source`

`tla-generator` asks a model for a TLA+ spec, then loops: check it, read back
what TLC did not like, hand that to the model as feedback, ask again.
`tlakit.check_source` is the "check it" step -- a spec as a string in,
a `CheckResult` out, no file on disk required.

This notebook drives the rest of that loop without a live model. In its
place is a fixed sequence of attempts, each fixing the bug the previous
attempt's diagnostics point at -- a syntax slip, then a logic bug, then a
spec that passes. That is deliberately the honest way to make a notebook
CI runs on every push behave the same on every run: no network call, no
API key, and no dependence on what a model feels like generating today.


## The attempts

A `Counter` module that is supposed to stay in `0..5`. Each string below is
what a model might hand back after seeing the previous attempt's
diagnostics.

* **draft** -- `==` where TLA+ wants `=` inside an expression. SANY refuses
  to parse it.
* **unbounded** -- parses, but `Increment` never stops, so the bound
  invariant fails almost immediately.
* **fixed** -- `Increment` is guarded by the bound, and a `Stutter` action
  keeps the spec from deadlocking once it is reached.


In [ ]:
ATTEMPTS = [
    (
        "draft",
        """---- MODULE Counter ----
EXTENDS Naturals

VARIABLES count

Init == count = 0
Increment == count' == count + 1
Spec == Init /\\ [][Increment]_count

BoundedBelowFive == count =< 5
====
""",
    ),
    (
        "unbounded",
        """---- MODULE Counter ----
EXTENDS Naturals

VARIABLES count

Init == count = 0
Increment == count' = count + 1
Spec == Init /\\ [][Increment]_count

BoundedBelowFive == count =< 5
====
""",
    ),
    (
        "fixed",
        """---- MODULE Counter ----
EXTENDS Naturals

VARIABLES count

Init == count = 0
Increment == count < 5 /\\ count' = count + 1
Stutter == count = 5 /\\ UNCHANGED count
Next == Increment \\/ Stutter
Spec == Init /\\ [][Next]_count

BoundedBelowFive == count =< 5
====
""",
    ),
]


## Formatting diagnostics for the next prompt

`CheckResult.diagnostics` is a list of `Diagnostic` objects; each already
renders as `module:line:column: message` through `__str__`. `feedback_for`
turns a whole result into the block of text `tla-generator` would splice
into its next prompt to the model: what went wrong, and -- when TLC found a
counterexample rather than a parse error -- the trace that shows it.


In [ ]:
import tlakit


def feedback_for(result: tlakit.CheckResult) -> str:
    """Render a CheckResult as feedback for the next generation attempt."""
    if result.ok:
        return "TLC found no violation. The spec is accepted as-is."

    lines = [f"TLC outcome: {result.outcome.value}"]
    for diagnostic in result.diagnostics:
        lines.append(f"- {diagnostic}")

    if result.trace is not None:
        lines.append(f"Counterexample trace ({len(result.trace)} states):")
        for i, state in enumerate(result.trace.states):
            changed = sorted(result.trace.delta(i)) if i else sorted(state)
            lines.append(f"  {i + 1}. changed={changed} {state}")

    return "\n".join(lines)


`feedback_for` only touches the dataclasses in `tlakit.result`, so it runs
the same whether the `CheckResult` came from a real TLC run or was built by
hand -- worth checking once against a hand-built one before trusting it in
a loop that does depend on TLC.


In [ ]:
_sample = tlakit.CheckResult(
    outcome=tlakit.Outcome.PARSE_ERROR,
    diagnostics=[
        tlakit.Diagnostic(
            tlakit.Severity.ERROR,
            'Encountered "Beginning of definition" at line 7, column 14 and token "=="',
            module="Counter",
            line=7,
            column=14,
        )
    ],
    trace=None,
    stats=tlakit.Stats(),
    raw=tlakit.RawOutput(argv=[], exit_code=1, stdout="", stderr=""),
)

rendered = feedback_for(_sample)
print(rendered)
assert "Counter:7:14" in rendered


## Running the loop

Each attempt goes through `check_source` exactly as `tla-generator` would
call it; the printed feedback is what would go back into the next prompt.
The loop stops at the first attempt that passes.

Model checking needs `tla2tools.jar`. When it is not on this machine,
`check_source` raises `tlakit.JarNotFound` on the first call -- caught below
so the notebook still finishes cleanly; the diagnostics formatting above
already proved out the part that does not need Java.


In [ ]:
def run_loop(attempts):
    for retries, (label, source) in enumerate(attempts):
        print(f"=== attempt: {label} ===")
        result = tlakit.check_source(source, invariants=["BoundedBelowFive"])
        print(feedback_for(result))
        print()
        if result.ok:
            return label, retries, result
    raise AssertionError("ran out of attempts without a passing spec")


try:
    winning_label, retries, final = run_loop(ATTEMPTS)
except tlakit.JarNotFound as exc:
    winning_label, retries, final = None, None, None
    print(f"tla2tools.jar not found ({exc}); skipping the live loop.")

if final is not None:
    assert winning_label == "fixed"
    assert final.outcome is tlakit.Outcome.OK
    print(f"Loop converged on attempt {winning_label!r} after {retries} retries.")
